In [121]:
from typing import Annotated
from typing_extensions import TypedDict
from pprint import pprint

import numexpr
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt
from llmsource import llm


PDF_PATH = "chapter8.pdf"
THREAD_ID = "rag_model_hitl"


print("Loading PDF...")
loader = PyMuPDFLoader(PDF_PATH)
documents = loader.load()
print(f"Documents loaded: {len(documents)}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.split_documents(documents)
print(f"Chunks created: {len(chunks)}")

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("FAISS database created successfully.")


@tool
def search_pdf(query: str) -> str:
    """Search the PDF knowledge base for information relevant to the user's question."""
    docs = retriever.invoke(query)
    results = []

    for doc in docs:
        page = doc.metadata.get("page")
        page_number = page + 1 if page is not None else "Unknown"
        results.append(f"Page {page_number}\n{doc.page_content}")

    return "\n\n---\n\n".join(results)


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = numexpr.evaluate(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


tools = [search_pdf, calculator]
tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

system_message = SystemMessage(
    content="""
You are a technical assistant with access to a PDF knowledge base.

For any technical question, request the search_pdf tool before answering.
Do not answer technical questions from your own knowledge.
Use calculator only for mathematical calculations.

If the human user rejects a requested tool call, do not call the same tool again
for the same request. Explain that you cannot complete that part without approval.
""".strip()
)


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    tool_approved: bool | None


def chat_node(state: ChatState) -> ChatState:
    messages = [system_message, *state["messages"]]
    response = llm_with_tools.invoke(messages)
    return {
        "messages": [response],
        "tool_approved": None,
    }


def route_after_chat(state: ChatState):
    last_message = state["messages"][-1]

    if getattr(last_message, "tool_calls", None):
        return "human_approval"

    return END


def human_approval_node(state: ChatState):
    last_message = state["messages"][-1]
    tool_calls = last_message.tool_calls
    requested_tools = [
        {
            "tool": tool_call["name"],
            "arguments": tool_call["args"],
        }
        for tool_call in tool_calls
    ]

    approved = interrupt(
        {
            "question": "Do you approve these tool calls?",
            "tool_calls": requested_tools,
        }
    )

    if approved:
        return {"tool_approved": True}

    rejection_messages = [
        ToolMessage(
            content="The human user rejected this tool execution.",
            tool_call_id=tool_call["id"],
            name=tool_call["name"],
        )
        for tool_call in tool_calls
    ]

    return {
        "tool_approved": False,
        "messages": rejection_messages,
    }


def route_after_approval(state: ChatState):
    if state["tool_approved"]:
        return "tools"

    return "chat_node"


graph_builder = StateGraph(ChatState)
graph_builder.add_node("chat_node", chat_node)
graph_builder.add_node("human_approval", human_approval_node)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chat_node")
graph_builder.add_conditional_edges("chat_node", route_after_chat)
graph_builder.add_conditional_edges("human_approval", route_after_approval)
graph_builder.add_edge("tools", "chat_node")

checkpoint = MemorySaver()
chatbot = graph_builder.compile(checkpointer=checkpoint)

config = {
    "configurable": {
        "thread_id": THREAD_ID,
    }
}


def ask_yes_no(prompt: str) -> bool:
    while True:
        decision = input(prompt).strip().lower()

        if decision in {"yes", "y"}:
            return True

        if decision in {"no", "n"}:
            return False

        print("Please enter yes or no.")


def continue_with_human_approval(response):
    while "__interrupt__" in response:
        interrupt_data = response["__interrupt__"][0].value

        print("\n" + "=" * 60)
        print("HUMAN APPROVAL REQUIRED")
        print("=" * 60)
        print(interrupt_data["question"])
        print("\nRequested tool calls:")

        for tool_call in interrupt_data["tool_calls"]:
            print(f"\nTool: {tool_call['tool']}")
            print(f"Arguments: {tool_call['arguments']}")

        approved = ask_yes_no("\nApprove tool execution? (yes/no): ")
        response = chatbot.invoke(Command(resume=approved), config=config)

    return response


user_query = input("Enter your question: ").strip()

if not user_query:
    raise ValueError("Please enter a question before running the notebook cell.")

initial_state = {
    "messages": [HumanMessage(content=user_query)],
    "tool_approved": None,
}

response = chatbot.invoke(initial_state, config=config)
response = continue_with_human_approval(response)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(response["messages"][-1].content)

print("\n" + "=" * 60)
print("MESSAGES")
print("=" * 60)
for message in response["messages"]:
    print(f"\n{type(message).__name__}")
    print(message.content)

    if getattr(message, "tool_calls", None):
        print("Tool calls:")
        pprint(message.tool_calls)


Loading PDF...
Documents loaded: 91
Chunks created: 185
FAISS database created successfully.

HUMAN APPROVAL REQUIRED
Do you approve these tool calls?

Requested tool calls:

Tool: search_pdf
Arguments: {'query': 'signal coupling'}

FINAL ANSWER
Based on the search results, here is what I know about **signal coupling**:

**Definition:**
Signal coupling occurs when electrical signals transfer from one wire (or set of wires) to another due to them being too close together. This is particularly problematic for signal integrity, especially between AC power conductors and low-level instrument signal wiring (such as thermocouple or pH sensor cables).

**Two Main Mechanisms:**
1.  **Capacitive Coupling:**
    *   Occurs due to the natural capacitance between insulated wires separated by a dielectric (insulating substance).
    *   This forms a "bridge" for AC signals to cross between wires.
    *   The strength of this coupling is inversely proportional to capacitive reactance ($X_C = \frac{1